In [0]:
%load_ext autoreload
%autoreload 2
# Enables autoreload; learn more at https://docs.databricks.com/en/files/workspace-modules.html#autoreload-for-python-modules
# To disable autoreload; run %autoreload 0

# Initialization

In [0]:
import pyspark.sql.functions as F
from pyspark.sql.functions import col, trim
from pyspark.sql.types import StringType
from pyspark.sql.window import Window
import bike_lakehouse.cleaning_data as hf

# Read Bronze Table


In [0]:
# I am reading from the source which is the Bronze
df_bronze=spark.table("workspace.bronze.crm_cust_info")

In [0]:
df_bronze.show(10)

# Silver Transformations

In [0]:
RENAME_MAP = {
    "cst_id": "customer_id",
    "cst_key": "customer_number",
    "cst_firstname": "first_name",
    "cst_lastname": "last_name",
    "cst_marital_status": "marital_status",
    "cst_gndr": "gender",
    "cst_create_date": "created_date"
}

##  Trimming 
 

In [0]:
# Calling trim function from cleaning.py file
df_silver=hf.trim_string_columns(df_bronze)

In [0]:
df_silver.display()

## Name Normalization

In [0]:
df_silver = (df_silver.withColumn(
    "cst_gndr",
    F.when(F.upper(F.col("cst_gndr")) == "M", "Male")
     .when(F.upper(F.col("cst_gndr")) == "F", "Female")
     .otherwise("Unknown")
)
.withColumn(
    "cst_marital_status",
    F.when(F.upper(F.col("cst_marital_status")) == "S", "Single")
     .when(F.upper(F.col("cst_marital_status")) == "M", "Married")
     .otherwise("Unknown"))
)
display(df_silver.limit(10))


## Renaming The Columns

In [0]:
df_silver=hf.rename_columns(df_silver,RENAME_MAP)
df_silver.show(5)

## Remove Records with Missing Customer ID

In [0]:
# Check if there are NULL values in the customer_id column
df_silver.filter(col("last_name").isNull()).show()

In [0]:
df_silver=df_silver.na.drop(subset=["customer_id"])
# df = df.filter(col("customer_id").isNotNull())

## Check NULLs

In [0]:
# Counts NULL values for each column in a DataFrame.
null_stats=hf.nulls_count_per_column(df_silver)
null_stats.display()


## Check Duplicates 

In [0]:
df_silver.groupBy("customer_id").count().filter(col("count") > 1).display()

In [0]:
window=Window.partitionBy("customer_id").orderBy(F.col("created_date").desc())
df_silver=df_silver.withColumn("flag",F.row_number().over(window)).filter(col("flag")==1).drop("flag") # flag=1--> “the latest row per customer”

display(df_silver)



In [0]:
# Counts the number of rows and columns after the deduplication.
hf.row_col_count(df_silver)

# Sanity checks

In [0]:
# Counts NULL values for each column in a DataFrame.
null_stats=hf.nulls_count_per_column(df_silver)
null_stats.display()

In [0]:
#sanity check
df_silver.limit(10).display()

# Write Into Silver Table
# 

In [0]:
df_silver.write.mode("overwrite").format("delta").saveAsTable("workspace.silver.crm_customers")

In [0]:
%sql
SELECT * FROM silver.crm_customers
LIMIT 10
